# 01 — Images as tensors

**Day 1 · Notebook 1 of 4**

Goal: by the end of this notebook you can answer

1. What *is* an image, as far as PyTorch is concerned?
2. What does `(N, C, H, W)` mean and why does the order matter?
3. What does normalization do to pixel values, and how do you undo it for display?

Run every cell in order. Stop and inspect the **shape** of every tensor you create — that habit alone prevents most CV bugs.

In [ ]:
import sys, pathlib
# Make the local src/cvcourse package importable when running notebooks
# from the notebooks/ folder.
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import torch
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from cvcourse import show_image, show_grid

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

## 1. Load an image — three ways

Pretend you have never used PyTorch. An image on disk is just bytes. Different
libraries hand you different representations of the same picture.

Real life: you'll get an image as a `PIL.Image`, a numpy array, or a torch
tensor — same picture, three representations. Before touching any real data,
let's construct a *synthetic* image so we control every pixel.

In [ ]:
# Build a synthetic 8x8 RGB image by hand so we can reason about every value.
img_np = np.zeros((8, 8, 3), dtype=np.uint8)
img_np[:, :4, 0] = 255     # left half is red
img_np[:, 4:, 2] = 255     # right half is blue
img_np[3:5, :, 1] = 255    # middle horizontal stripe is green
print("numpy image shape (H, W, C):", img_np.shape)
print("dtype:", img_np.dtype, "min:", img_np.min(), "max:", img_np.max())

plt.imshow(img_np); plt.title("synthetic 8x8 RGB"); plt.axis("off"); plt.show()

## 2. HWC vs CHW — the layout switch that bites everyone

NumPy and PIL store images as **H × W × C**. PyTorch convolutions want **C × H × W**.
Get this wrong and a model trains on garbage without ever raising an error.

In [ ]:
img_t = torch.from_numpy(img_np)          # still HWC
print("HWC:", img_t.shape)

img_chw = img_t.permute(2, 0, 1)          # -> CHW
print("CHW:", img_chw.shape)

# Sanity check: the red channel should be 255 on the left and 0 on the right.
print("red row 0:", img_chw[0, 0].tolist())

### Mini-exercise

Without running the cell, predict the shape after each step:

```python
x = torch.zeros(28, 28)        # ?
x = x.unsqueeze(0)             # ?
x = x.unsqueeze(0)             # ?
x = x.repeat(4, 1, 1, 1)       # ?
```

Then run it and check.

In [ ]:
x = torch.zeros(28, 28)
print(x.shape)
x = x.unsqueeze(0); print(x.shape)
x = x.unsqueeze(0); print(x.shape)
x = x.repeat(4, 1, 1, 1); print(x.shape)

## 3. Batches: the leading dimension

PyTorch models almost always expect a batch dimension up front: `(N, C, H, W)`.
Even when you have one image, you wrap it in a batch of size 1.

In [ ]:
batch = img_chw.unsqueeze(0).float() / 255.0     # 1 image, float in [0, 1]
print("batch:", batch.shape, batch.dtype, "min/max:", batch.min().item(), batch.max().item())

## 4. From uint8 [0, 255] to float [0, 1] to normalized

Most pretrained networks were trained with inputs roughly centered around zero.
The classic ImageNet normalization is

```
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
x_norm = (x - mean) / std
```

Display works on the *un-normalized* image. Forgetting to invert normalization
before plotting is a top-3 source of "why does my image look insane" bugs.

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

x = batch[0]                       # (3, 8, 8) in [0, 1]
x_norm = (x - mean) / std
x_back = x_norm * std + mean       # invert
print("after normalize  -> mean/std per channel:",
      x_norm.mean(dim=(1, 2)).tolist(), x_norm.std(dim=(1, 2)).tolist())
print("after un-normalize -> equals original?",
      torch.allclose(x_back, x, atol=1e-6))

## 5. Resize, crop, the usual tricks

`torchvision.transforms.v2` is the modern API. It works on PIL images **and**
tensors. Use tensor transforms whenever you can — they're faster and
GPU-friendly.

In [ ]:
from torchvision.transforms import v2 as T

big = torch.rand(3, 256, 256)
tfm = T.Compose([
    T.Resize(64),                # shortest side -> 64
    T.CenterCrop(48),
    T.ToDtype(torch.float32, scale=True),
])
out = tfm(big)
print("input:", big.shape, "-> output:", out.shape)

## 6. Visualize a batch

Visualizing batches *constantly* is the #1 trick for CV. If you can't picture
what's flowing through your model, you'll lose hours to bugs that a single
`plt.imshow` would have caught.

In [ ]:
fake_batch = torch.rand(8, 3, 32, 32)
show_grid(fake_batch, cols=4)

## Recap

- Images are tensors. NumPy/PIL use **HWC**, PyTorch uses **CHW**.
- Batches are **(N, C, H, W)**.
- Always check `.shape`, `.dtype`, and value range.
- Normalize for the network. Un-normalize for the eyes.

Next up: **convolutions** — the operation that makes CNNs work.